# Notebook 2 - Data Loading, Audit & Cleaning## Cookie Cats A/B Testing**Phase:** CRISP-DM Phase 2 & 3 - Data Understanding & Preparation> **Navigation:** <- `01_business_understanding.ipynb` | Next -> `03_web_scraping.ipynb`---

## Step 0 - Environment Setup

In [ ]:
import sys, os, warningswarnings.filterwarnings('ignore')sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12})os.makedirs('../reports/figures', exist_ok=True)print("Environment ready.")

---## Step 1 - Data Loading & Initial Exploration

In [2]:
from processing import load_data, explore_data

df = load_data()
info = explore_data(df)

print(f"\nDataset Shape: {info['shape']}")
print(f"Columns: {info['columns']}")
print(f"Duplicates: {info['duplicates']}")
print(f"\nVersion Distribution:")
for k, v in info['version_distribution'].items():
    print(f"  {k}: {v:,}")

Dataset loaded: 90,189 rows × 5 columns

Dataset Shape: (90189, 5)
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']
Duplicates: 0

Version Distribution:
  gate_40: 45,489
  gate_30: 44,700


In [3]:
# Show first few rows and dtypes
print("First 5 rows:")
print(df.head().to_string())
print(f"\nData Types:\n{df.dtypes}")
print(f"\nBasic Statistics:\n{df.describe().to_string()}")

First 5 rows:
   userid  version  sum_gamerounds  retention_1  retention_7
0     116  gate_30               3        False        False
1     337  gate_30              38         True        False
2     377  gate_40             165         True        False
3     483  gate_40               1        False        False
4     488  gate_40             179         True         True

Data Types:
userid             int64
version           object
sum_gamerounds     int64
retention_1         bool
retention_7         bool
dtype: object

Basic Statistics:
             userid  sum_gamerounds
count  9.018900e+04    90189.000000
mean   4.998412e+06       51.872457
std    2.883286e+06      195.050858
min    1.160000e+02        0.000000
25%    2.512230e+06        5.000000
50%    4.995815e+06       16.000000
75%    7.496452e+06       51.000000
max    9.999861e+06    49854.000000


---## Step 2 - Data Quality AuditA rigorous audit checks schema integrity, missing values, and outlier detection **before** any cleaning.

In [4]:
from processing import data_audit

audit_results = data_audit(df)

# Display outlier summary
print(f"\n📊 Audit Summary:")
print(f"  Schema valid: {audit_results['schema']['schema_valid']}")
print(f"  Missing cells: {audit_results['missing_values']['total_missing_cells']}")
print(f"  IQR outliers: {audit_results['outliers_iqr']['outlier_count']:,} "
      f"({audit_results['outliers_iqr']['outlier_pct']}%)")
print(f"  Z-score outliers (sum_gamerounds): {audit_results['outliers_zscore']['outlier_counts'].get('sum_gamerounds', 0):,}")

  DATA QUALITY AUDIT

  Schema valid        : True
  Total missing cells : 0
  Exact duplicate rows: 0
  Duplicate user IDs  : 0

  IQR outliers (sum_gamerounds):
    Q1=5, Q3=51, IQR=46
    Bounds: [-64, 120]
    Outliers: 10,177 (11.28%)

  Z-score outliers (|z| > 3):
    userid                        : 0
    sum_gamerounds                : 425

  Value ranges:
    sum_gamerounds: [0, 49854]  (negative: 0)


📊 Audit Summary:
  Schema valid: True
  Missing cells: 0
  IQR outliers: 10,177 (11.28%)
  Z-score outliers (sum_gamerounds): 425


---## Step 3 - Data CleaningCap outliers at the 99th percentile based on audit findings.

In [5]:
from processing import preprocess_data

df_clean = preprocess_data(df, cap_outliers=True, cap_percentile=0.99)
df_clean.describe()

Capped 898 extreme game-round values at 99% percentile (493 rounds).
After cleaning: 90,189 rows


,userid,sum_gamerounds,retention_1,retention_7
count,9.018900e+04,90189.000000,90189.000000,90189.000000
mean,4.998412e+06,48.993647,0.445210,0.186065
std,2.883286e+06,84.205426,0.496992,0.389161
min,1.160000e+02,0.000000,0.000000,0.000000
25%,2.512230e+06,5.000000,0.000000,0.000000
50%,4.995815e+06,16.000000,0.000000,0.000000
75%,7.496452e+06,51.000000,1.000000,0.000000
max,9.999861e+06,493.000000,1.000000,1.000000


In [6]:
from processing import save_processed_data

# Save clean (pre-augmentation) dataset to data/processed/
save_processed_data(df_clean, "cookie_cats_clean.csv")

💾 Saved processed data → d:\Data Science\src\..\data\processed\cookie_cats_clean.csv


'd:\\Data Science\\src\\..\\data\\processed\\cookie_cats_clean.csv'

---## Summary| Step | Action | Output ||------|--------|--------|| Load | `load_data()` | Raw DataFrame (90,189 x 5) || Audit | `data_audit()` | Schema + missing + outlier report || Clean | `preprocess_data()` | Outlier-capped DataFrame || Save | `save_processed_data()` | `data/processed/cookie_cats_clean.csv` |-> Next: `03_web_scraping.ipynb`